# NLP Literary Corpus Pipeline — Full Notebook
## Evaluating VADER Sentiment Analysis Accuracy Across Literary Genres in a Diverse EPUB Corpus

This notebook runs the **complete 7-step NLP pipeline** from EPUB loading to summarisation, all inside Jupyter. Run each cell in order from top to bottom.

---

**Research Question:** To what extent does automated VADER sentiment analysis accurately reflect the emotional register of literary genre in a diverse EPUB corpus of 99 texts published between 1818 and 1946?

**H1:** VADER scores will partially but not fully reflect genre-based emotional register expectations.  
**H2:** Greater misclassification in 19th-century texts than early 20th-century texts.  
**H3:** Non-English texts will show the highest VADER misclassification rates.

---

### How to run this notebook
1. Click the first cell
2. Press **Shift + Enter** to run it and move to the next
3. Or click **Kernel → Restart and Run All** to run everything at once

> ⚠️ Running all cells will take **3 to 4 hours** on CPU. Make sure your laptop is plugged in and sleep is disabled.


## Step 0 — Install and Import Dependencies

In [ ]:
# Run this cell first — installs everything needed
# Skip if you already ran pip install -r requirements.txt

import subprocess, sys

packages = [
    'nltk', 'spacy', 'gensim', 'scikit-learn',
    'transformers', 'EbookLib', 'beautifulsoup4',
    'sumy', 'lexicalrichness', 'matplotlib',
    'seaborn', 'pyLDAvis', 'wordcloud',
    'pandas', 'numpy', 'tqdm', 'scipy==1.11.4'
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], capture_output=True)

print('✓ All packages installed')


In [ ]:
# Download required NLTK data and spaCy model
import nltk
import subprocess, sys

for resource in ['punkt', 'punkt_tab', 'stopwords', 'vader_lexicon',
                 'averaged_perceptron_tagger', 'wordnet']:
    nltk.download(resource, quiet=True)

# Download spaCy model
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'],
               capture_output=True)

print('✓ NLTK data downloaded')
print('✓ spaCy model downloaded')


In [ ]:
# Core imports
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Pipeline settings
EPUB_DIR   = '../data'
OUTPUT_DIR = '../outputs'
PLOTS_DIR  = os.path.join(OUTPUT_DIR, 'plots')
SUMS_DIR   = os.path.join(OUTPUT_DIR, 'summaries')

for d in [OUTPUT_DIR, PLOTS_DIR, SUMS_DIR]:
    os.makedirs(d, exist_ok=True)

# Display settings
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_rows', 30)
sns.set_theme(style='whitegrid')

print('✓ Imports complete')
print(f'  Project root: {PROJECT_ROOT}')
print(f'  EPUB folder:  {os.path.abspath(EPUB_DIR)}')
print(f'  Output folder:{os.path.abspath(OUTPUT_DIR)}')
epub_count = len([f for f in os.listdir(EPUB_DIR) if f.endswith('.epub')]) if os.path.exists(EPUB_DIR) else 0
print(f'  EPUB files found: {epub_count}')


---
## Step 1 — Load EPUB Corpus

Parses all EPUB files in the `data/` folder using ebooklib and BeautifulSoup4. Each book is extracted chapter by chapter and stored as a structured dictionary.


In [ ]:
from src.epub_extract import load_corpus

print('Loading EPUB corpus...')
t0 = time.time()

documents = load_corpus(EPUB_DIR)

print(f'\n✓ Loaded {len(documents)} documents in {time.time()-t0:.1f}s')
print()

# Display corpus summary table
corpus_info = pd.DataFrame([{
    'Title'  : d['title'][:50],
    'Author' : d['author'][:30],
    'Chapters': len(d['chapters']),
    'Characters': f"{len(d['full_text']):,}"
} for d in documents])

display(corpus_info)


---
## Step 2 — Descriptive Statistics

Computes vocabulary richness (TTR, MTLD, HD-D) and sentence length statistics for each text.


In [ ]:
from src.descriptive import corpus_stats
from src.visualise import plot_lexical_diversity, plot_sentence_histogram

print('Computing descriptive statistics...')
print('(This may take 2-3 minutes for 99 texts)')
t0 = time.time()

stats_df = corpus_stats(documents)
stats_df.to_csv(os.path.join(OUTPUT_DIR, 'corpus_stats.csv'), index=False)

print(f'\n✓ Statistics computed in {time.time()-t0:.1f}s')
print(f'  Saved to: outputs/corpus_stats.csv')


In [ ]:
# Display statistics table
display(stats_df[['title', 'num_tokens', 'ttr', 'mtld', 'hdd', 'mean_sent_len']].round(4))


In [ ]:
# Generate and display lexical diversity chart
plot_lexical_diversity(stats_df, os.path.join(PLOTS_DIR, 'lexical_diversity.png'))
plot_sentence_histogram(stats_df, os.path.join(PLOTS_DIR, 'sentence_length.png'))

display(Image(filename=os.path.join(PLOTS_DIR, 'lexical_diversity.png'), width=900))
display(Image(filename=os.path.join(PLOTS_DIR, 'sentence_length.png'), width=900))
print('✓ Charts saved')


---
## Step 3 — Sentiment Analysis ⭐ Primary Method

This is the core step for answering the research question. VADER compound scores are computed for every sentence in every text.

> ⚠️ This step processes 99 texts and will take approximately **30-45 minutes**.


In [ ]:
from src.sentiment import vader_chapters, full_sentiment, sentiment_summary
from src.visualise import plot_sentiment_arc, plot_sentiment_bars

print('Running VADER sentiment analysis...')
print('This will take approximately 30-45 minutes. Please wait.')
print()

t0 = time.time()
sent_summaries = []

for i, doc in enumerate(documents):
    title = doc['title']
    print(f'  [{i+1:>2}/{len(documents)}] {title[:50]}')

    # Safe filename
    safe = title[:20]
    for ch in ['/', '\\', ':', '?', '*', '"', '<', '>', '|']:
        safe = safe.replace(ch, '')
    safe = safe.replace(' ', '_')

    # Chapter sentiment arc
    arc = vader_chapters(doc['chapters'])
    plot_sentiment_arc(arc, title,
        os.path.join(PLOTS_DIR, f'sentiment_arc_{safe}.png'))

    # Sentence-level sentiment (first 50k chars)
    sent_df = full_sentiment(doc['full_text'][:50_000], use_bert=False)
    sent_df['title'] = title
    sent_df.to_csv(os.path.join(OUTPUT_DIR, f'sentiment_{safe}.csv'), index=False)

    summary = sentiment_summary(sent_df)
    summary['title'] = title
    sent_summaries.append(summary)

sent_summary_df = pd.DataFrame(sent_summaries)
sent_summary_df.to_csv(os.path.join(OUTPUT_DIR, 'sentiment_summary.csv'), index=False)
plot_sentiment_bars(sent_summary_df, os.path.join(PLOTS_DIR, 'sentiment_bars.png'))

print(f'\n✓ Sentiment analysis complete in {time.time()-t0:.1f}s')
print(f'  Saved to: outputs/sentiment_summary.csv')


In [ ]:
# Display sentiment results
sent_sorted = sent_summary_df.sort_values('mean_vader_compound', ascending=False)
display(sent_sorted[['title', 'mean_vader_compound', 'pct_positive_vader', 'pct_negative_vader']].round(4))


In [ ]:
# Display sentiment bars chart
display(Image(filename=os.path.join(PLOTS_DIR, 'sentiment_bars.png'), width=900))

# Key finding for research question
most_pos = sent_summary_df.loc[sent_summary_df['mean_vader_compound'].idxmax()]
most_neg = sent_summary_df.loc[sent_summary_df['mean_vader_compound'].idxmin()]

print('\n=== KEY FINDING FOR RESEARCH QUESTION ===')
print(f'Most positive text: {most_pos["title"]}')
print(f'  Score: {most_pos["mean_vader_compound"]:+.4f}')
print(f'Most negative text: {most_neg["title"]}')
print(f'  Score: {most_neg["mean_vader_compound"]:+.4f}')


In [ ]:
# Display sentiment arc for Frankenstein — key anomaly
import glob
arc_files = glob.glob(os.path.join(PLOTS_DIR, 'sentiment_arc_Frankenstein*.png'))
if arc_files:
    print('Sentiment Arc — Frankenstein (expected: NEGATIVE, actual: see below)')
    display(Image(filename=arc_files[0], width=900))

arc_files2 = glob.glob(os.path.join(PLOTS_DIR, 'sentiment_arc_Peter*.png'))
if arc_files2:
    print('Sentiment Arc — Peter Pan (expected: POSITIVE)')
    display(Image(filename=arc_files2[0], width=900))


---
## Step 4 — Named Entity Recognition

Identifies persons, places, organisations and dates across all 99 texts. Provides supporting evidence for genre-based structural differences.

> ⚠️ This step will take approximately **45-60 minutes** for 99 texts.


In [ ]:
from src.ner import ner_corpus, top_entities, extract_entities
from src.visualise import plot_ner_bars, plot_ner_heatmap

print('Running Named Entity Recognition...')
print('This will take approximately 45-60 minutes. Please wait.')
t0 = time.time()

ner_df = ner_corpus(documents, model='en_core_web_sm')
ner_df.to_csv(os.path.join(OUTPUT_DIR, 'ner_results.csv'), index=False)

top_chars = {}
for doc in documents:
    ents = extract_entities(doc['full_text'], model='en_core_web_sm')
    top  = top_entities(ents, label='PERSON', n=10)
    top_chars[doc['title']] = top

with open(os.path.join(OUTPUT_DIR, 'top_characters.json'), 'w') as f:
    json.dump(top_chars, f, indent=2)

plot_ner_bars(ner_df,    os.path.join(PLOTS_DIR, 'ner_bars.png'))
plot_ner_heatmap(ner_df, os.path.join(PLOTS_DIR, 'ner_heatmap.png'))

print(f'\n✓ NER complete in {time.time()-t0:.1f}s')
print(f'  Saved to: outputs/ner_results.csv')


In [ ]:
# Display NER results
display(ner_df[['title', 'PERSON', 'GPE/LOC', 'ORG', 'DATE/TIME', 'OTHER', 'total']].head(20))

display(Image(filename=os.path.join(PLOTS_DIR, 'ner_bars.png'), width=900))


---
## Step 5 — TF-IDF Keyword Extraction

Identifies the most distinctive vocabulary for each text relative to the whole corpus. The keywords serve as a genre fingerprint.


In [ ]:
from src.tfidf_keywords import tfidf_pipeline
from src.visualise import plot_tfidf_wordcloud, plot_tfidf_bars

print('Running TF-IDF keyword extraction...')
t0 = time.time()

kw_df, kw_dict, _, _ = tfidf_pipeline(documents, n_keywords=20)
kw_df.to_csv(os.path.join(OUTPUT_DIR, 'tfidf_keywords.csv'), index=False)

titles = [d['title'] for d in documents]
for title in titles:
    safe = title[:20]
    for ch in ['/', '\\', ':', '?', '*', '"', '<', '>', '|']:
        safe = safe.replace(ch, '')
    safe = safe.replace(' ', '_')

    top5 = [kw for kw, _ in kw_dict.get(title, [])[:5]]
    print(f'  {title[:40]:<40} {top5}')

    plot_tfidf_wordcloud(kw_dict, title,
        os.path.join(PLOTS_DIR, f'wordcloud_{safe}.png'))
    plot_tfidf_bars(kw_df, title,
        os.path.join(PLOTS_DIR, f'tfidf_bars_{safe}.png'))

print(f'\n✓ TF-IDF complete in {time.time()-t0:.1f}s')
print(f'  Saved to: outputs/tfidf_keywords.csv')


In [ ]:
# Display TF-IDF results table
display(kw_df[kw_df['rank'] <= 5].head(50))


---
## Step 6 — LDA Topic Modelling

Discovers 20 latent thematic topics across the 99-text corpus without human labelling. Evaluates whether genre-coherent clusters emerge automatically.

> ⚠️ This step will take approximately **20-30 minutes**.


In [ ]:
from src.lda_topics import lda_pipeline
from src.visualise import plot_coherence_curve, plot_topic_heatmap

print('Running LDA topic modelling (K=20)...')
print('This will take approximately 20-30 minutes. Please wait.')
t0 = time.time()

lda_result = lda_pipeline(
    documents,
    auto_k=False,
    num_topics=20,
    output_dir=OUTPUT_DIR,
)
lda_result['doc_topic_matrix'].to_csv(
    os.path.join(OUTPUT_DIR, 'doc_topic_matrix.csv'), index=False)

with open(os.path.join(OUTPUT_DIR, 'lda_topics.txt'), 'w') as f:
    for t in lda_result['topic_words']:
        words = ', '.join(w for w, _ in t['words'])
        f.write(f"Topic {t['topic_id']:>2}: {words}\n")

plot_topic_heatmap(lda_result['doc_topic_matrix'],
                   os.path.join(PLOTS_DIR, 'topic_heatmap.png'))

print(f'\n✓ LDA complete in {time.time()-t0:.1f}s')
print(f'  Saved to: outputs/doc_topic_matrix.csv')
print(f'  Interactive visualisation: outputs/lda_vis.html')


In [ ]:
# Display discovered topics
print('=== DISCOVERED LDA TOPICS ===\n')
for t in lda_result['topic_words']:
    words = ', '.join(w for w, _ in t['words'])
    print(f"Topic {t['topic_id']:>2}: {words}")


In [ ]:
# Display topic heatmap
display(Image(filename=os.path.join(PLOTS_DIR, 'topic_heatmap.png'), width=900))

# Display document-topic matrix
dtm = lda_result['doc_topic_matrix']
topic_cols = [c for c in dtm.columns if c.startswith('topic_')]
dtm['dominant_topic'] = dtm[topic_cols].idxmax(axis=1)
dtm['dominant_score'] = dtm[topic_cols].max(axis=1).round(3)
display(dtm[['title', 'dominant_topic', 'dominant_score']].head(30))


---
## Step 7 — Text Summarisation

Generates extractive TextRank summaries for all 99 texts. Provides a human-readable reference point for interpreting VADER sentiment scores.


In [ ]:
from src.summarise import summarise_corpus

print('Running TextRank summarisation...')
t0 = time.time()

sum_df = summarise_corpus(documents, method='textrank', n_sentences=10)
sum_df.to_csv(os.path.join(OUTPUT_DIR, 'summaries.csv'), index=False)

for _, row in sum_df.iterrows():
    safe = row['title'][:30]
    for ch in ['/', '\\', ':', '?', '*', '"', '<', '>', '|']:
        safe = safe.replace(ch, '')
    safe = safe.replace(' ', '_')
    out_file = os.path.join(SUMS_DIR, f'{safe}_summary.txt')
    with open(out_file, 'w', encoding='utf-8') as f:
        f.write(f"TITLE : {row['title']}\n")
        f.write(f"AUTHOR: {row['author']}\n")
        f.write(f"METHOD: {row['method']}\n\n")
        f.write(row['summary'])

print(f'\n✓ Summarisation complete in {time.time()-t0:.1f}s')
print(f'  Saved to: outputs/summaries/')
print(f'  Files created: {len(sum_df)}')


In [ ]:
# Display summary for Frankenstein
frank_sum = sum_df[sum_df['title'].str.contains('Frankenstein', case=False, na=False)]
if len(frank_sum) > 0:
    print('=== FRANKENSTEIN SUMMARY ===')
    print(frank_sum['summary'].values[0][:800])
    print()

# Display summary for Peter Pan
peter_sum = sum_df[sum_df['title'].str.contains('Peter Pan', case=False, na=False)]
if len(peter_sum) > 0:
    print('=== PETER PAN SUMMARY ===')
    print(peter_sum['summary'].values[0][:800])


---
## Step 8 — Research Question Evaluation

This section directly answers the research question and evaluates all three hypotheses using the pipeline outputs.


In [ ]:
print('=' * 65)
print('  RESEARCH QUESTION EVALUATION')
print('=' * 65)
print()
print('RQ: To what extent does VADER accurately reflect the')
print('    emotional register of literary genre?')
print()

sent = pd.read_csv(os.path.join(OUTPUT_DIR, 'sentiment_summary.csv'))

total = len(sent)
pos   = (sent['mean_vader_compound'] >  0.05).sum()
neg   = (sent['mean_vader_compound'] < -0.05).sum()
neu   = total - pos - neg

print(f'CORPUS SENTIMENT PROFILE ({total} texts):')
print(f'  Positive texts : {pos:>3}  ({pos/total*100:.1f}%)')
print(f'  Negative texts : {neg:>3}  ({neg/total*100:.1f}%)')
print(f'  Neutral texts  : {neu:>3}  ({neu/total*100:.1f}%)')
print(f'  Mean compound  : {sent["mean_vader_compound"].mean():+.4f}')
print()

most_pos = sent.loc[sent['mean_vader_compound'].idxmax()]
most_neg = sent.loc[sent['mean_vader_compound'].idxmin()]
print(f'Most positive: {most_pos["title"][:50]}  ({most_pos["mean_vader_compound"]:+.4f})')
print(f'Most negative: {most_neg["title"][:50]}  ({most_neg["mean_vader_compound"]:+.4f})')


In [ ]:
print('=' * 65)
print('  HYPOTHESIS EVALUATION')
print('=' * 65)
print()

# H1
print('H1: VADER will partially but not fully reflect genre expectations')
print()

# Check gothic horror
gothic_keywords = ['Dracula', 'Frankenstein', 'Dorian Gray', 'Jekyll',
                   'Yellow Wallpaper', 'Carmilla', 'Ghost Stories', 'Beetle']
gothic_match = sent[sent['title'].str.contains(
    '|'.join(gothic_keywords), case=False, na=False)]

children_keywords = ['Peter Pan', 'Secret Garden', 'Wind in the Willows',
                     'Pollyanna', 'Daddy-Long-Legs', 'Railway Children', 'Heidi']
children_match = sent[sent['title'].str.contains(
    '|'.join(children_keywords), case=False, na=False)]

if len(gothic_match) > 0:
    g_mean = gothic_match['mean_vader_compound'].mean()
    direction = 'NEGATIVE ✓ (as expected)' if g_mean < 0 else 'POSITIVE ✗ (unexpected)'
    print(f"  Gothic/Horror texts ({len(gothic_match)} matched):")
    print(f"    Mean VADER: {g_mean:+.4f} → {direction}")
    for _, r in gothic_match.iterrows():
        print(f"      {r['title'][:45]:<45} {r['mean_vader_compound']:+.4f}")
    print()

if len(children_match) > 0:
    c_mean = children_match['mean_vader_compound'].mean()
    direction = 'POSITIVE ✓ (as expected)' if c_mean > 0 else 'NEGATIVE ✗ (unexpected)'
    print(f"  Children's Fiction texts ({len(children_match)} matched):")
    print(f"    Mean VADER: {c_mean:+.4f} → {direction}")
    for _, r in children_match.iterrows():
        print(f"      {r['title'][:45]:<45} {r['mean_vader_compound']:+.4f}")


In [ ]:
# H3 — Non-English texts
print('H3: Non-English texts will show highest misclassification rates')
print()

french_keywords = ['Pour comprendre', 'Phantom of the Opera', 'Cyrano',
                   'Germinal', 'Twenty Thousand', 'Mysterious Island',
                   'Journey to the Centre', 'Arsene Lupin', 'Yellow Room']
german_keywords  = ['Faust', 'Siddhartha', 'Beyond Good', 'Zarathustra',
                    'Heidi', 'Emil', 'Relativity']

french_match  = sent[sent['title'].str.contains('|'.join(french_keywords), case=False, na=False)]
german_match  = sent[sent['title'].str.contains('|'.join(german_keywords), case=False, na=False)]
non_eng_match = pd.concat([french_match, german_match]).drop_duplicates()
eng_match     = sent[~sent.index.isin(non_eng_match.index)]

print(f'  Non-English texts found: {len(non_eng_match)}')
if len(non_eng_match) > 0:
    ne_pct_pos = (non_eng_match['pct_positive_vader']).mean()
    e_pct_pos  = (eng_match['pct_positive_vader']).mean()
    print(f'  Mean % positive — Non-English: {ne_pct_pos:.1f}%')
    print(f'  Mean % positive — English:     {e_pct_pos:.1f}%')
    print()
    for _, r in non_eng_match.iterrows():
        print(f"    {r['title'][:45]:<45} {r['mean_vader_compound']:+.4f}  "
              f"(+{r['pct_positive_vader']:.1f}% / -{r['pct_negative_vader']:.1f}%)")


---
## Step 9 — Pipeline Complete

All steps have finished. Your results are saved in the `outputs/` folder.


In [ ]:
print('=' * 65)
print('  PIPELINE COMPLETE')
print('=' * 65)
print()

outputs = {
    'corpus_stats.csv'    : 'Lexical diversity and sentence stats',
    'sentiment_summary.csv': 'VADER scores for all 99 texts',
    'ner_results.csv'     : 'Named entity distributions',
    'tfidf_keywords.csv'  : 'Top 20 keywords per text',
    'doc_topic_matrix.csv': 'LDA topic probabilities',
    'lda_topics.txt'      : '20 discovered topic descriptions',
    'lda_vis.html'        : 'Interactive topic visualisation',
    'summaries/'          : 'Per-text TextRank summaries',
    'plots/'              : f'{len(os.listdir(PLOTS_DIR))} charts',
}

print('Key output files:')
for fname, desc in outputs.items():
    fpath = os.path.join(OUTPUT_DIR, fname)
    exists = '✓' if os.path.exists(fpath) else '✗'
    print(f'  {exists}  {fname:<30}  {desc}')

print()
print('Open outputs/lda_vis.html in your browser for')
print('interactive topic exploration.')
print()
print('All results are ready for your project report.')
